In [ ]:
import torch
from transformers import AutoTokenizer

print("PyTorch version:", torch.__version__)


PyTorch version: 2.11.0+cpu


In [ ]:
# Cell 1: Install Required Libraries
!pip install -q transformers datasets accelerate torch
print("✅ Libraries Installed Successfully!")


✅ Libraries Installed Successfully!


1: Tokenizer Under the Hood

In [ ]:
from transformers import AutoTokenizer

# 1. Load standard BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
text = "Sahil is unhappily debugging code using transformers!"

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
# 2. Shabdon ke tukde (Sub-words) dekhte hain
tokens = tokenizer.tokenize(text)
print("1. Sub-words (Tokens):")
print(tokens)
print("-" * 50)


1. Sub-words (Tokens):
['sa', '##hil', 'is', 'un', '##ha', '##pp', '##ily', 'de', '##bu', '##gging', 'code', 'using', 'transformers', '!']
--------------------------------------------------


In [ ]:
# 3. Har token ka unique Number (ID) dekhte hain
input_ids = tokenizer.convert_tokens_to_ids(tokens)
print("2. Token IDs (Numbers):")
print(input_ids)
print("-" * 50)


2. Token IDs (Numbers):
[7842, 19466, 2003, 4895, 3270, 9397, 6588, 2139, 8569, 12588, 3642, 2478, 19081, 999]
--------------------------------------------------


In [ ]:

# 4. Numbers ko wapas English me decode karke dekhte hain
decoded_text = tokenizer.decode(input_ids)
print("3. Reconstructed Text (Decode):")
print(decoded_text)
print("-" * 50)

3. Reconstructed Text (Decode):
sahil is unhappily debugging code using transformers!
--------------------------------------------------


In [ ]:
# 5. Tokenizer ki Total Dictionary (Vocab) Size
print(f"4. Total Vocabulary Size: {tokenizer.vocab_size} tokens")

4. Total Vocabulary Size: 30522 tokens


 2: Special Tokens & Attention Mask ka Live Switch

In [ ]:
#  Padding, Special Tokens & Attention Mask Live
sentences = [
    "Hello Sahil!",  # Chota sentence (2 words)
    "Deep learning with transformers is completely changing artificial intelligence.",  # Lamba sentence
]

In [ ]:
# Tokenizer ko direct call karte hain with padding=True aur PyTorch tensors
batch = tokenizer(sentences, padding=True, return_tensors="pt")
print("1. Tensors ki Shape (Batch Size, Sequence Length):")
print(batch["input_ids"].shape)
print("-" * 60)
print("2. Sentence 1 ke input_ids (Dhyan se aakhri ke numbers dekho):")
print(batch["input_ids"][0])
print("\nSentence 1 ka attention_mask (Dhyan se 1 aur 0 dekho):")
print(batch["attention_mask"][0])
print("-" * 60)
print("3. Sentence 1 ko Decode karke dekhte hain (Special Tokens dikhenge):")
print(tokenizer.decode(batch["input_ids"][0]))
print("-" * 60)
print("4. Sentence 2 ka attention_mask:")
print(batch["attention_mask"][1])


1. Tensors ki Shape (Batch Size, Sequence Length):
torch.Size([2, 12])
------------------------------------------------------------
2. Sentence 1 ke input_ids (Dhyan se aakhri ke numbers dekho):
tensor([  101,  7592,  7842, 19466,   999,   102,     0,     0,     0,     0,
            0,     0])

Sentence 1 ka attention_mask (Dhyan se 1 aur 0 dekho):
tensor([1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0])
------------------------------------------------------------
3. Sentence 1 ko Decode karke dekhte hain (Special Tokens dikhenge):
[CLS] hello sahil! [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
------------------------------------------------------------
4. Sentence 2 ka attention_mask:
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])


 3: Static vs Dynamic Padding Showdown (Live Comparison)

In [ ]:
# Cell 4: Static Padding vs Dynamic DataCollator
from transformers import DataCollatorWithPadding
# -------------------------------------------------------------
# 1. PEHLE DEKHO: Static Padding (Har sample ko 128 tak kheencha)
# -------------------------------------------------------------
static_batch = tokenizer(
    sentences, padding="max_length", max_length=128, return_tensors="pt"
)
print("❌ STATIC PADDING RESULT:")
print(f"Tensor Shape: {static_batch['input_ids'].shape}")
print(f"Total numbers in Matrix: {static_batch['input_ids'].numel()}")
print(
    f"Sentence 1 me real tokens: 4 | Faltu ke [PAD] tokens: "
    f"{128 - 4} tokens!"
)
print("=" * 60)

❌ STATIC PADDING RESULT:
Tensor Shape: torch.Size([2, 128])
Total numbers in Matrix: 256
Sentence 1 me real tokens: 4 | Faltu ke [PAD] tokens: 124 tokens!


In [ ]:
# 2. AB DEKHO: Dynamic Padding (DataCollatorWithPadding)
# -------------------------------------------------------------
# Pehle sentences ko bina padding ke tokenize kiya
sample1 = tokenizer(sentences[0])
sample2 = tokenizer(sentences[1])
# Hugging Face ka smart collator bulaya
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
# Collator ne dono samples ko lekar smart batch banaya
dynamic_batch = data_collator([sample1, sample2])
print("✅ DYNAMIC PADDING RESULT (DataCollatorWithPadding):")
print(f"Tensor Shape: {dynamic_batch['input_ids'].shape}")
print(f"Total numbers in Matrix: {dynamic_batch['input_ids'].numel()}")
print("-" * 60)
print("💡 Farak dekha?")
print(
    f"Static Matrix size tha {static_batch['input_ids'].numel()} numbers."
    f" Dynamic Matrix size hai sirf {dynamic_batch['input_ids'].numel()}"
    " numbers!"
)
print(
    "👉 GPU ko 90% kam numbers process karne pade — Isiliye Dynamic Padding"
    " rocket ki tarah fast hoti hai!"
)

✅ DYNAMIC PADDING RESULT (DataCollatorWithPadding):
Tensor Shape: torch.Size([2, 12])
Total numbers in Matrix: 24
------------------------------------------------------------
💡 Farak dekha?
Static Matrix size tha 256 numbers. Dynamic Matrix size hai sirf 24 numbers!
👉 GPU ko 90% kam numbers process karne pade — Isiliye Dynamic Padding rocket ki tarah fast hoti hai!


4: Real Healthcare Dataset

In [ ]:
# Cell 5: Real-world Dataset Loading with Apache Arrow
from datasets import load_dataset
print("⏳ Loading ChatDoctor Dataset (100k+ rows)...")
dataset = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k")
print("\n✅ Dataset Successfully Loaded!")
print("-" * 60)


⏳ Loading ChatDoctor Dataset (100k+ rows)...


README.md:   0%|          | 0.00/542 [00:00<?, ?B/s]

data/train-00000-of-00001-5e7cb295b9cff0(…): reconstructing file:   0%|          |  0.00B / 70.5MB            

data/train-00000-of-00001-5e7cb295b9cff0(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/112165 [00:00<?, ? examples/s]


✅ Dataset Successfully Loaded!
------------------------------------------------------------


In [ ]:
# 1. Dataset structure dekhte hain
print("1. Dataset Structure & Splits:")
print(dataset)
print("-" * 60)


1. Dataset Structure & Splits:
DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 112165
    })
})
------------------------------------------------------------


In [ ]:
# 2. Total kitne samples hain training set me
total_rows = len(dataset["train"])
print(f"2. Total Training Samples: {total_rows:,} rows")
print("-" * 60)

2. Total Training Samples: 112,165 rows
------------------------------------------------------------


In [ ]:
# 3. Pehle sample ka Patient Query aur Doctor Answer dekhte hain
sample = dataset["train"][0]
print("3. Sample Patient Query (Input):")
print(sample["input"])
print("-" * 60)
print("4. Sample Doctor Answer (Output):")
print(sample["output"])


3. Sample Patient Query (Input):
I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walking unsteadily, as i tried to focus i feel nauseous. I try to vomit but it wont come out.. After taking panadol and sleep for few hours, i still feel the same.. By the way, if i lay down or sit down, my head do not spin, only when i want to move around then i feel the whole world is spinning.. And it is normal stomach discomfort at the same time? Earlier after i relieved myself, the spinning lessen so i am not sure whether its connected or coincidences.. Thank you doc!
------------------------------------------------------------
4. Sample Doctor Answer (Output):
Hi, Thank you for posting your query. The most likely cause for your symptoms is benign paroxysmal positional vertigo (BPPV), a type of peripheral vertigo. In this condition, the most common symptom is dizziness or giddiness, which is made worse with movements. Accompanying nausea and vo

5: High-Speed .map(batched=True) Preprocessing Pipeline! 🎯

In [ ]:
# Cell 6: Fast Tokenization Pipeline with .map(batched=True)
# Quick experiment ke liye pehle 2,000 samples lete hain (Super fast testing ke liye)
small_train_data = dataset["train"].select(range(2000))

In [ ]:
# 1. Ashish Sir ka Tokenize Function
def tokenize_function(examples):
  # Patient query aur Doctor answer ko ek stream me combine kiya
  combined_text = [
      inp + " " + out for inp, out in zip(examples["input"], examples["output"])
  ]
  # Tokenize with truncation and max_length
  return tokenizer(
      combined_text, truncation=True, padding="max_length", max_length=512
  )
print("🚀 Preprocessing & Tokenizing 2,000 samples with .map(batched=True)...")


🚀 Preprocessing & Tokenizing 2,000 samples with .map(batched=True)...


In [ ]:
# 2. .map() run karte hain (Dhyan se remove_columns dekho)
tokenized_dataset = small_train_data.map(
    tokenize_function,
    batched=True,  # 1000 rows ek sath parallel process hongi
    remove_columns=[
        "input",
        "output",
        "instruction",
    ],  # Raw text hataya taaki PyTorch khush rahe
)
print("\n✅ Tokenization Complete!")
print("-" * 60)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]


✅ Tokenization Complete!
------------------------------------------------------------


In [ ]:
# 3. Naye Preprocessed Dataset ke Features dekhte hain
print("1. Final Preprocessed Dataset Structure:")
print(tokenized_dataset)
print("-" * 60)

1. Final Preprocessed Dataset Structure:
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2000
})
------------------------------------------------------------


In [ ]:
# 4. Pehle sample ke input_ids aur attention_mask check karte hain
sample_tokenized = tokenized_dataset[0]
print(f"2. Total Tokens in Sample 0: {len(sample_tokenized['input_ids'])}")
print(f"3. First 15 input_ids: {sample_tokenized['input_ids'][:15]}")
print(f"4. First 15 attention_mask: {sample_tokenized['attention_mask'][:15]}")
print("-" * 60)
print(
    "🎉 BADHAI HO! Yeh Dataset ab direct kisi bhi LLM ko Fine-Tune karne ke"
    " liye 100% READY hai!"
)


2. Total Tokens in Sample 0: 512
3. First 15 input_ids: [101, 1045, 8271, 2039, 2023, 2851, 3110, 1996, 2878, 2282, 2003, 9419, 2043, 1045, 2001]
4. First 15 attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
------------------------------------------------------------
🎉 BADHAI HO! Yeh Dataset ab direct kisi bhi LLM ko Fine-Tune karne ke liye 100% READY hai!


Model & DataCollator Load karna

In [5]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
)

# 1. Chota sa model load kar rahe hain
model_name = "distilbert/distilgpt2"
model = AutoModelForCausalLM.from_pretrained(modal_name)
tokenizer = AutoTokenizer.from_pretrained(modal_name)
print("✅ Model Successfully Loaded!")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

✅ Model Successfully Loaded!


In [6]:
# GPT-2 ke End token ko hi PAD token bana rahe hain
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id
# Causal LM ke liye mlm=False collator ready kiya
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

TrainingArguments

In [7]:
from transformers import TrainingArguments
import torch
training_args = TrainingArguments(
    output_dir="./my_test_results",  # Model save hone ka folder
    per_device_train_batch_size=2,  # GPU ko ek baar me sirf 2 samples
    gradient_accumulation_steps=2,  # Virtual Batch size = 2 x 2 = 4
    learning_rate=2e-5,  # Choti speed taaki purana na bhoole
    max_steps=20,  # Quick test ke liye sirf 20 steps
    logging_steps=5,  # Har 5 step par screen par loss print ho
    report_to="none",  # Koi faltu popup na aaye
    fp16=torch.cuda.is_available(),  # Agar GPU ho toh 2x fast speed on ho
)
print("✅ TrainingArguments Ready!")

✅ TrainingArguments Ready!


 The Trainer Engine & The Training Run

In [9]:
from transformers import Trainer, AutoModelForCausalLM, AutoTokenizer, DataCollatorForLanguageModeling
from datasets import load_dataset

# Safely load the model and tokenizer if they were not defined due to runtime restart
try:
    model
    tokenizer
except NameError:
    print("⚠️ Model or tokenizer not found in memory. Loading them now...")
    model_name = "distilbert/distilgpt2"
    model = AutoModelForCausalLM.from_pretrained(model_name)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.pad_token_id

# Safely load dataset and preprocess if missing from memory
try:
    tokenized_dataset
    data_collator
except NameError:
    print("⚠️ Preprocessed dataset or data collator not found. Re-creating them now...")
    dataset = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k")
    small_train_data = dataset["train"].select(range(2000))

    def tokenize_function(examples):
        combined_text = [
            inp + " " + out for inp, out in zip(examples["input"], examples["output"])
        ]
        return tokenizer(
            combined_text, truncation=True, padding="max_length", max_length=512
        )

    tokenized_dataset = small_train_data.map(
        tokenize_function,
        batched=True,
        remove_columns=["input", "output", "instruction"]
    )
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 1. Gym Trainer ko saari cheezein pakdayi
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("🚀 Starting Training Loop... Dhyan se screen par Loss dekhna!")

# 2. Asli jaadui button!
trainer.train()

⚠️ Preprocessed dataset or data collator not found. Re-creating them now...


README.md:   0%|          | 0.00/542 [00:00<?, ?B/s]

data/train-00000-of-00001-5e7cb295b9cff0(…): reconstructing file:   0%|          |  0.00B / 70.5MB            

data/train-00000-of-00001-5e7cb295b9cff0(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/112165 [00:00<?, ? examples/s]

...

🚀 Starting Training Loop... Dhyan se screen par Loss dekhna!


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
5,4.538984
10,4.455872
15,4.292179
20,4.574321


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=20, training_loss=4.465339088439942, metrics={'train_runtime': 16.8123, 'train_samples_per_second': 4.758, 'train_steps_per_second': 1.19, 'total_flos': 10451870023680.0, 'train_loss': 4.465339088439942, 'epoch': 0.04})